# Branch 1 RD/RA GCS Retraining — Colab Notebook



In [ ]:
# 1. Update the gcloud CLI (Google Cloud SDK) to ensure the latest 'gcloud storage' features are available
!echo "deb [signed-by=/usr/share/keyrings/cloud.google.gpg] https://packages.cloud.google.com/apt cloud-sdk main" | sudo tee -a /etc/apt/sources.list.d/google-cloud-sdk.list
!curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key --keyring /usr/share/keyrings/cloud.google.gpg add -
!sudo apt-get update && sudo apt-get install -y google-cloud-cli

!gcloud --version

## 2. Authenticate and configure Google Cloud


In [ ]:
# @title
import os, subprocess, pathlib
from google.colab import auth
import os, subprocess

auth.authenticate_user()

# Core GCS / Colab paths
PROJECT = "fluent-webbing-496616-u8"
BUCKET = "miamioh-resa-data"
GCS_MOUNT_POINT = "/content/gcs"
WORK_ROOT = "/content/work"
CODE_ROOT = "/content/work/code"
MOUNT_ROOT = pathlib.Path(GCS_MOUNT_POINT)
MOUNT_ROOT.mkdir(parents=True, exist_ok=True)

GCLOUD_PROCESS_COUNT = 4
GCLOUD_THREAD_COUNT = 16


# Install gcsfuse if missing.
if subprocess.run(["bash", "-lc", "command -v gcsfuse"], capture_output=True).returncode != 0:
    subprocess.run(["bash", "-lc", "export GCSFUSE_REPO=gcsfuse-$(lsb_release -c -s); echo deb https://packages.cloud.google.com/apt $GCSFUSE_REPO main | tee /etc/apt/sources.list.d/gcsfuse.list"], check=True)
    subprocess.run(["bash", "-lc", "curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | apt-key add -"], check=True)
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "gcsfuse"], check=True)
subprocess.run(["gcloud", "components", "update", "--quiet"], check=False)
subprocess.run(["gcloud", "config", "set", "project", PROJECT], check=True)
subprocess.run(["gcloud", "config", "set", "storage/process_count", str(GCLOUD_PROCESS_COUNT)], check=True)
subprocess.run(["gcloud", "config", "set", "storage/thread_count", str(GCLOUD_THREAD_COUNT)], check=True)

subprocess.run(["gcsfuse", "--implicit-dirs", BUCKET, str(MOUNT_ROOT)], check=False)
print("Mounted path candidate:", MOUNT_ROOT / "CapstoneData")
print("Configured project:", PROJECT)
print("Configured bucket:", BUCKET)


### run only once after GSC mount

In [ ]:
!python -m pip install -q --upgrade pip
!python -m pip install -q pandas


## write train script in background

In [ ]:
# @title
%%writefile /content/branch1_rdra_gcs_retrain.py
from __future__ import annotations

import os
import shlex
import subprocess
import sys
from pathlib import Path

DEFAULT_BUCKET = "miamioh-resa-data"
DEFAULT_CODE_ROOT = Path("/content/work/code")
CODE_SYNC_EXCLUDE = r"LLM_ML/data/.*|data/.*|colab_outputs/.*|__pycache__/.*|\.ipynb_checkpoints/.*"
TARGET_DRIVER_REL = Path("branch1/training/curated_gcs_retrain_driver.py")


def _arg_value(flag: str, default: str) -> str:
    argv = sys.argv[1:]
    for idx, token in enumerate(argv):
        if token == flag and idx + 1 < len(argv):
            return argv[idx + 1]
        if token.startswith(flag + "="):
            return token.split("=", 1)[1]
    return default


def _stage_args() -> list[str]:
    stages: list[str] = []
    for token in sys.argv[1:]:
        if token.startswith("-"):
            break
        stages.append(token)
    return stages or ["rolling"]


def _sync_code(bucket: str, code_root: Path, dry_run: bool) -> None:
    code_root.mkdir(parents=True, exist_ok=True)
    cmd = [
        "gcloud",
        "--quiet",
        "--verbosity=error",
        "storage",
        "rsync",
        f"gs://{bucket}/CapstoneData/code_v2/RESA_mmWave",
        str(code_root),
        "--recursive",
        "--delete-unmatched-destination-objects",
        "--exclude",
        CODE_SYNC_EXCLUDE,
    ]
    print("$", " ".join(shlex.quote(part) for part in cmd), flush=True)
    if dry_run:
        return
    subprocess.run(cmd, check=True)


def _dispatch(target: Path, code_root: Path, argv: list[str]) -> int:
    env = os.environ.copy()
    env["PYTHONPATH"] = str(code_root) + ((":" + env["PYTHONPATH"]) if env.get("PYTHONPATH") else "")
    env.setdefault("RESA_CODE_ROOT", str(code_root))
    cmd = [sys.executable, str(target), *argv]
    print("$", " ".join(shlex.quote(part) for part in cmd), flush=True)
    return subprocess.run(cmd, env=env).returncode


def main() -> int:
    stages = _stage_args()
    bucket = _arg_value("--bucket", DEFAULT_BUCKET)
    code_root = Path(_arg_value("--code-root", str(DEFAULT_CODE_ROOT)))
    dry_run = "--dry-run" in sys.argv[1:]
    target = code_root / TARGET_DRIVER_REL

    if "setup-code" in stages or not target.exists():
        _sync_code(bucket, code_root, dry_run)

    forwarded = sys.argv[1:]
    if "setup-code" in stages:
        removed = False
        cleaned: list[str] = []
        for token in forwarded:
            if not removed and token == "setup-code":
                removed = True
                continue
            cleaned.append(token)
        forwarded = cleaned
        remaining_stages = [stage for stage in stages if stage != "setup-code"]
        if not remaining_stages:
            print("Bootstrap sync complete.", flush=True)
            return 0

    if not target.exists():
        raise FileNotFoundError(f"Missing curated driver after sync: {target}")
    return _dispatch(target, code_root, forwarded)


if __name__ == "__main__":
    raise SystemExit(main())


## 5. Shared run configuration


In [ ]:
# @title Shared run configuration — curated manifest Branch 1 four-class retraining
from pathlib import Path
import os
import shlex
import subprocess
import time

DRIVER = Path("/content/branch1_rdra_gcs_retrain.py")
RUN_LOG_DIR = Path("/content/work/debug_driver_logs")
RUN_LOG_DIR.mkdir(parents=True, exist_ok=True)

# Helps reduce CUDA allocator fragmentation on large-batch A100 runs.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Curated-session intake.
# Live catalog: gs://miamioh-resa-data/CapstoneData/curated/manifests/session_catalog.csv
# Session payloads: gs://miamioh-resa-data/CapstoneData/curated/sessions/<tier>/<dataset_id>/<session_id>.tar.gz
TRAIN_QUALITY_TIERS = ["gold"]
EXTERNAL_VAL_QUALITY_TIERS = ["gold"]
TRAIN_SPLITS = ["train"]
EXTERNAL_VAL_SPLITS = ["external_val", "holdout", "val", "test"]
TRAIN_DATASET_IDS = []  # optional allowlist; leave empty to use every matching curated train session
EXTERNAL_VAL_DATASET_IDS = []  # optional allowlist; leave empty to use every matching curated eval session
DATASET_GROUPS = TRAIN_DATASET_IDS or ["auto"]  # local staging/output naming only
EXTERNAL_VAL_GROUP = "external_val"  # local staging/output naming only
SESSION_PREFIX = "session_"

# Transfer / rolling-batch settings. These mostly affect CPU/RAM/disk, not GPU VRAM.
SESSION_BATCH_SIZE = 30
SESSION_DOWNLOAD_BATCH_SIZE = 10
GCLOUD_PROCESS_COUNT = 4
GCLOUD_THREAD_COUNT = 16

# Training schedule.
TRAIN_EPOCHS = 30
TRAIN_EPOCHS_PER_SESSION_BATCH = 5

# A100-heavy throughput knobs. Increase TRAIN_WINDOWS_PER_BATCH first if GPU is still underused.
TRAIN_WINDOWS_PER_BATCH = 256
TRAIN_NUM_WORKERS = 8
TRAIN_PIN_MEMORY = True

# A100-heavy model/input-capacity knobs. Set to 0 to omit a knob and use the training script default.
# These are appended only if train_hybrid_rd_patch_kpconv.py exposes the matching CLI flag.
TRAIN_POINTS_PER_FRAME = 128
TRAIN_EMB_DIMS = 512
TRAIN_TEMPORAL_LAYERS = 4
TRAIN_TEMPORAL_HEADS = 8
TRAIN_GATE_HIDDEN = 256
TRAIN_RD_PATCH_EMBED_DIM = 64
TRAIN_K = 32
TRAIN_N_KERNEL_POINTS = 31

# Feature/output toggles.
DISABLE_GHOST_TEACHER = False
COPY_GHOST_ASSETS = False
FORCE_REBUILD_EXTERNAL_VAL = False
COMPACT_TRAIN_OUTPUT = True
DEBUG = True
DRY_RUN = False
MAX_BATCHES = None  # set to a small integer for debugging, e.g. 1


def build_driver_cmd(*stages, max_batches=MAX_BATCHES):
    cmd = [
        "python3", str(DRIVER),
        *stages,
        "--bucket", BUCKET,
        "--project", PROJECT,
        "--gcs-mount-point", GCS_MOUNT_POINT,
        "--work-root", WORK_ROOT,
        "--code-root", CODE_ROOT,
        "--session-batch-size", str(SESSION_BATCH_SIZE),
        "--session-download-batch-size", str(SESSION_DOWNLOAD_BATCH_SIZE),
        "--gcloud-process-count", str(GCLOUD_PROCESS_COUNT),
        "--gcloud-thread-count", str(GCLOUD_THREAD_COUNT),
        "--train-epochs", str(TRAIN_EPOCHS),
        "--train-epochs-per-session-batch", str(TRAIN_EPOCHS_PER_SESSION_BATCH),
        "--train-windows-per-batch", str(TRAIN_WINDOWS_PER_BATCH),
        "--train-num-workers", str(TRAIN_NUM_WORKERS),
        "--train-points-per-frame", str(TRAIN_POINTS_PER_FRAME),
        "--train-emb-dims", str(TRAIN_EMB_DIMS),
        "--train-temporal-layers", str(TRAIN_TEMPORAL_LAYERS),
        "--train-temporal-heads", str(TRAIN_TEMPORAL_HEADS),
        "--train-gate-hidden", str(TRAIN_GATE_HIDDEN),
        "--train-rd-patch-embed-dim", str(TRAIN_RD_PATCH_EMBED_DIM),
        "--train-k", str(TRAIN_K),
        "--train-n-kernel-points", str(TRAIN_N_KERNEL_POINTS),
        "--dataset-groups", ",".join(DATASET_GROUPS),
        "--external-val-group", EXTERNAL_VAL_GROUP,
        "--session-prefix", SESSION_PREFIX,
        "--train-quality-tiers", ",".join(TRAIN_QUALITY_TIERS),
        "--external-val-quality-tiers", ",".join(EXTERNAL_VAL_QUALITY_TIERS),
        "--train-splits", ",".join(TRAIN_SPLITS),
        "--external-val-splits", ",".join(EXTERNAL_VAL_SPLITS),
    ]

    if TRAIN_DATASET_IDS:
        cmd += ["--train-dataset-ids", ",".join(TRAIN_DATASET_IDS)]
    if EXTERNAL_VAL_DATASET_IDS:
        cmd += ["--external-val-dataset-ids", ",".join(EXTERNAL_VAL_DATASET_IDS)]
    if TRAIN_PIN_MEMORY:
        cmd.append("--train-pin-memory")
    if DISABLE_GHOST_TEACHER:
        cmd.append("--disable-ghost-teacher")
    if COPY_GHOST_ASSETS:
        cmd.append("--copy-ghost-assets")
    if FORCE_REBUILD_EXTERNAL_VAL:
        cmd.append("--force-rebuild-external-val")
    if COMPACT_TRAIN_OUTPUT:
        cmd.append("--compact-train-output")
    if DEBUG:
        cmd.append("--debug")
    if DRY_RUN:
        cmd.append("--dry-run")
    if max_batches is not None:
        cmd += ["--max-batches", str(max_batches)]
    return [str(x) for x in cmd]


def run_driver(*stages, max_batches=MAX_BATCHES):
    cmd = build_driver_cmd(*stages, max_batches=max_batches)
    log_path = RUN_LOG_DIR / f"driver_{'_'.join(stages) or 'run'}_{time.strftime('%Y%m%d_%H%M%S')}.log"
    print("$", " ".join(shlex.quote(x) for x in cmd))
    print("Driver log:", log_path)

    env = os.environ.copy()
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert proc.stdout is not None
    tail = []
    with log_path.open("w", encoding="utf-8") as fh:
        fh.write("$ " + " ".join(shlex.quote(x) for x in cmd) + "\n")
        for line in proc.stdout:
            fh.write(line)
            fh.flush()
            print(line, end="")
            tail.append(line.rstrip("\n"))
            if len(tail) > 250:
                tail = tail[-250:]
    rc = proc.wait()
    if rc != 0:
        print("\n" + "=" * 78)
        print(f"Driver failed with return code {rc}")
        print("Full log:", log_path)
        print("Last 250 output lines:")
        for line in tail:
            print(line)
        raise subprocess.CalledProcessError(rc, cmd)
    return log_path


print("Curated Branch 1 four-class filters:")
print("  TRAIN_QUALITY_TIERS        =", TRAIN_QUALITY_TIERS)
print("  EXTERNAL_VAL_QUALITY_TIERS =", EXTERNAL_VAL_QUALITY_TIERS)
print("  TRAIN_SPLITS               =", TRAIN_SPLITS)
print("  EXTERNAL_VAL_SPLITS        =", EXTERNAL_VAL_SPLITS)
print("  TRAIN_DATASET_IDS          =", TRAIN_DATASET_IDS or "<all matching>")
print("  EXTERNAL_VAL_DATASET_IDS   =", EXTERNAL_VAL_DATASET_IDS or "<all matching>")
print("Session prefix:", repr(SESSION_PREFIX))
print("A100 hybrid RD training config:")
print("  TRAIN_WINDOWS_PER_BATCH =", TRAIN_WINDOWS_PER_BATCH)
print("  TRAIN_NUM_WORKERS       =", TRAIN_NUM_WORKERS)
print("  TRAIN_POINTS_PER_FRAME  =", TRAIN_POINTS_PER_FRAME)
print("  TRAIN_EMB_DIMS          =", TRAIN_EMB_DIMS)
print("  TRAIN_TEMPORAL          =", f"{TRAIN_TEMPORAL_LAYERS} layers / {TRAIN_TEMPORAL_HEADS} heads")
print("  TRAIN_K / KERNEL_POINTS =", TRAIN_K, TRAIN_N_KERNEL_POINTS)


## 6. Sync code from GCS

This bootstrap cell syncs the refactored repository from:

`gs://<bucket>/CapstoneData/code_v2/RESA_mmWave`

into `/content/work/code`, then dispatches into the repo-backed curated Branch 1 driver module.


In [ ]:
run_driver("setup-code")


## 9. Rolling retraining

This is the main workflow. For a quick debugging run, set `MAX_BATCHES = 1` in the configuration cell and re-run that cell before running this one.


In [ ]:
run_driver("rolling", max_batches=MAX_BATCHES)
